In [2]:
import pandas as pd
import pint
import pint_xarray
import xarray as xr

from util import create_region_graph, convert_to_xarray, aggregate_regions, extract_population
from constants import target_regions, region_mapping

from imagematerials.constants import _IMAGE_REGIONS as image_regions

In [3]:
ureg = pint.UnitRegistry()
ureg.define('terakilometer = 1e12 * kilometer')
ureg.define('milkilometer = 1e6 * kilometer')

pkmGlobal_original = pd.read_csv("image_3_4_data/pkmGlobalTot.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0)
tkmGlobal_original = pd.read_csv("image_3_4_data/tkmGlobalTot.csv", delimiter=";", usecols=range(8),skiprows=3).drop(0)
load_original = pd.read_csv("image_3_4_data/loadfactor.csv", delimiter=";", usecols=range(9),skiprows=3).drop(0) #

IMAGE_SE_original = pd.read_csv("image_3_4_data/SSP2_SE_IMAGE.csv", delimiter=";")
load_original = load_original.drop(["Unnamed: 2", "Unnamed: 3"], axis = 1)

In [4]:
pkmGlobal = convert_to_xarray(pkmGlobal_original, value_name = "pkm", variable = "mode")
tkmGlobal = convert_to_xarray(tkmGlobal_original, value_name = "tkm", variable = "mode")
load = convert_to_xarray(load_original, value_name = "load", variable = "mode")


In [5]:
# Convert units
pkmGlobal = pkmGlobal.pint.quantify('terakilometer') 
pkmGlobal= pkmGlobal.pint.to("km")

In [6]:
# Convert the region names in xr_image_output to match the target regions
knowledge_graph = create_region_graph()
xr_regions = pkmGlobal.coords["region"].values  # Extract current region names

xr_region_filter = knowledge_graph.find_relations_inverse(xr_regions, target_regions)

In [7]:
pkmGlobal_target_regions = aggregate_regions(pkmGlobal, xr_region_filter) #km
tkmGlobal_target_regions = aggregate_regions(tkmGlobal, xr_region_filter) # million km
load_target_regions = aggregate_regions(load, xr_region_filter, aggregation = 'mean') #person/vehicle

In [8]:
pop_yearly_million = pd.read_csv("image_3_4_data/pop_yearly.csv", delimiter=";", skiprows=3)
pop_yearly_million = pop_yearly_million.rename(columns=lambda x: int(x.replace("class_", "")) if "class_" in x else x)
pop_yearly_million = pop_yearly_million.set_index("t").stack().reset_index()
pop_yearly_million.columns = ["year", "region", "value"]  # Rename columns
pop_yearly_million = (
        pop_yearly_million.astype({"year": int, "region": int})  # Convert to integers
        .query("region not in [27, 28] and year in [2019, 2020, 2060]")  # Filter regions and years
    )
region_mapping = { i+1:region for i, region in enumerate(image_regions)}
pop_yearly_million["region"] = pop_yearly_million["region"].map(region_mapping)
pop_yearly_million_xr = pop_yearly_million.set_index(["region", "year"]).to_xarray()["value"]
pop_image_regions = pop_yearly_million_xr * 1e6 # from million to single people

pop = aggregate_regions(pop_image_regions, xr_region_filter)

In [9]:
tkmGlobal_target_regions = tkmGlobal_target_regions * 1e6 #from million tkm to tkm


In [10]:
pkm_per_capita = pkmGlobal_target_regions / pop
tkm_per_capita = tkmGlobal_target_regions / pop

In [11]:
# Assign names to the DataArrays
pkm_per_capita.name = 'pkm_per_capita'
tkm_per_capita.name = 'tkm_per_capita'
load_target_regions.name = 'load'

# Step 1: Convert xarrays to DataFrames
pkm_per_capita_df = pkm_per_capita.to_dataframe().reset_index()
tkm_per_capita_df = tkm_per_capita.to_dataframe().reset_index()
load_df = load_target_regions.to_dataframe().reset_index()

c:\Users\5982758\AppData\Local\anaconda3\envs\materials\lib\site-packages\xarray\core\variable.py:338: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


In [12]:
# Melt the pkm_per_capita DataFrame
pkm_per_capita_df_melted = pkm_per_capita_df.melt(id_vars=["region", "year", "mode"], 
                                                   value_vars=["pkm_per_capita"], 
                                                   var_name="variable", value_name="value")

# Assign the variable name for pkm_per_capita
pkm_per_capita_df_melted["variable"] = "pkm_per_capita"

# Melt the tkm_per_capita DataFrame
tkm_per_capita_df_melted = tkm_per_capita_df.melt(id_vars=["region", "year", "mode"], 
                                                   value_vars=["tkm_per_capita"], 
                                                   var_name="variable", value_name="value")

# Assign the variable name for tkm_per_capita
tkm_per_capita_df_melted["variable"] = "tkm_per_capita"

# Merge the two melted DataFrames
merged_df = pd.merge(pkm_per_capita_df_melted, tkm_per_capita_df_melted, 
                     on=["region", "year", "mode",'variable','value'], how="outer")

load_df_melted = load_df.melt(id_vars=["region", "year", "mode"], 
                              value_vars=["load"], 
                              var_name="variable", value_name="value")

# Assign the variable name for load
load_df_melted["variable"] = "load"

# Merge the two melted DataFrames
merged_df = pd.merge(merged_df, load_df_melted, 
                     on=["region", "year", "mode",'variable','value'], how="outer")

# Optionally, you may want to combine or rearrange columns as needed
#merged_df = merged_df[['region', 'year', 'mode', 'variable', 'value']]

merged_df.to_csv("output_data.csv", index=False)


In [13]:
merged_df[(merged_df['variable']=='pkm_per_capita')&(merged_df['mode']=='car')]

,region,year,mode,variable,value
4,Canada,2019,car,pkm_per_capita,28724.597296
22,Canada,2060,car,pkm_per_capita,28152.667987
40,Central Europe,2019,car,pkm_per_capita,9541.784756
58,Central Europe,2060,car,pkm_per_capita,11454.737191
76,China,2019,car,pkm_per_capita,2732.898477
94,China,2060,car,pkm_per_capita,3769.985655
112,India,2019,car,pkm_per_capita,945.328361
130,India,2060,car,pkm_per_capita,1657.925717
148,Japan,2019,car,pkm_per_capita,8094.866354
166,Japan,2060,car,pkm_per_capita,11214.119356


In [14]:
IMAGE_SE = IMAGE_SE_original[["Region","Variable","2020","2060"]]
IMAGE_SE.columns = IMAGE_SE.columns.str.lower()  # Convert column names to lowercase
total_pkm_tkm = IMAGE_SE[(IMAGE_SE['variable']=='Energy Service|Transportation|Passenger') | # billion pkm/yr
                    (IMAGE_SE['variable']=='Energy Service|Transportation|Freight')] # billion tkm/yr



total_pkm_tkm_melted = total_pkm_tkm.melt(id_vars=["region", "variable"], var_name="year", value_name="value")

total_pkm_tkm_melted = total_pkm_tkm_melted[total_pkm_tkm_melted["region"] != "World"]
total_pkm_tkm_melted["year"] = total_pkm_tkm_melted["year"].astype(int)

# Convert from billion km to km
total_pkm_tkm_melted["value"] *= 1e9

pkm_tkm_xr = total_pkm_tkm_melted.set_index(["region","year","variable"]).to_xarray()#["value"]

pkm_tkm_xr = pkm_tkm_xr.assign_coords(region=[xr_region_filter[r] for r in pkm_tkm_xr["region"].values])

pkm_tkm_xr = pkm_tkm_xr.groupby("region").sum()


pkm_tkm_per_capita = pkm_tkm_xr / pop

pkm_tkm_per_capita_df = pkm_tkm_per_capita.to_dataframe().reset_index()

pkm_tkm_per_capita_df.to_csv("pkm_tkm_per_capita_df.csv", index=False)

pkm_tkm_per_capita_df


,year,variable,region,value
0,2020,Energy Service|Transportation|Freight,Canada,74613.255505
1,2020,Energy Service|Transportation|Freight,Central Europe,41339.204121
2,2020,Energy Service|Transportation|Freight,China,14986.067197
3,2020,Energy Service|Transportation|Freight,India,2328.151150
4,2020,Energy Service|Transportation|Freight,Japan,40113.441836
5,2020,Energy Service|Transportation|Freight,Latin America,10001.163789
6,2020,Energy Service|Transportation|Freight,Middle East and Northern Africa,12891.815295
7,2020,Energy Service|Transportation|Freight,Other Asia,11245.877945
8,2020,Energy Service|Transportation|Freight,Other OECD,20347.841162
9,2020,Energy Service|Transportation|Freight,Reforming Economies,9437.543187
